# Лабораторная работа №11. TCP как разговор двух автоматов

Увидим handshake, sequence/acknowledgement numbers, затем payload. TCP рассматриваем как два согласованных state machine, а не как «надёжный UDP».

## Схема

![TCP](assets/tcp.svg)

`pc1` будет client, `pc2` — server на TCP/8080.

## 1. Подготовим адреса

`pc1`:

```bash
ip addr flush dev eth1
ip addr add 10.11.0.10/24 dev eth1
```

`pc2`:

```bash
ip addr flush dev eth1
ip addr add 10.11.0.20/24 dev eth1
```

Проверьте ping и затем переходите к transport.

### Q1. Зачем TCP handshake нужен обеим сторонам, если IP-связность уже подтверждена ping?

`введите ваш ответ`

## 2. Запустим server и client

На `pc2`:

```bash
nc -l -p 8080
```

На `pc1`:

```bash
nc 10.11.0.20 8080
```

После соединения введите несколько строк. В другом terminal можно смотреть:

```bash
ss -tn
```

In [ ]:
%%capture_traffic pc1:eth1 --filter "tcp.port == 8080" --timeout 15 --packets 200 --save captures/tcp.pcap
import time
print("Создайте TCP connection pc1 → pc2:8080 и отправьте одну строку")
time.sleep(13)

In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display
viewer = PcapViewer("captures/tcp.pcap", "tcp.port == 8080", limit=80)
frames = viewer.packets()
display(frames)
for n in frames["№"].head(8) if not frames.empty else []:
    print("\nframe", n)
    print("\n".join(viewer.packet(int(n), "protocols").splitlines()[:65]))

## 3. Handshake

Найдите три первых TCP segment:

```text
SYN
SYN, ACK
ACK
```

SYN занимает одно место в sequence space. Поэтому ACK на SYN подтверждает `initial sequence + 1`.

### Q2. Если SYN клиента имеет Seq=1000, какой Ack должен вернуть server и почему?

`введите ваш ответ`

## 4. Payload и следующий ACK

Найдите segment с application data. Сопоставьте `tcp.len`, Seq и следующий Ack.

Если segment начинается с Seq=N и несёт L байт payload, ожидаемый следующий byte — `N + L` (с учётом SYN/FIN, если они присутствуют).

### Q3. Segment несёт 20 байт payload начиная с Seq=5001. Какой Ack ожидается у peer, если потерь нет?

`введите ваш ответ`

## 5. Reliability — это state

TCP хранит sequence state, acknowledgements, receive window и retransmission logic. Потеря segment не означает немедленный разрыв connection: sender может повторить данные.

## CHECK

Checker проверяет адреса и сам выполняет короткий TCP exchange `pc1 → pc2:8080`, не полагаясь на ручной `nc` из notebook.

## Итог

TCP создаёт connection state до передачи application data. Sequence/Ack превращают byte stream в согласованный разговор, а window/retransmission управляют flow/reliability.